# Gate 2 · Part 1 — SQL (45 min, no reference): three queries, one with a partitioned window function

**Worked solutions.** Try each question yourself first (timer on, no notes), then compare.

**Data:** the three tidy tables from P1, in DuckDB (`P1 - NYC Restaurant Grades/data/processed/nyc_inspections.duckdb`, created by P1's `run_all.py`).

| Table | One row per | Columns used here |
| :-- | :-- | :-- |
| `restaurants` | restaurant | `camis`, `boro`, `cuisine` |
| `inspections` | inspection | `inspection_id`, `camis`, `inspection_date`, `program`, `kind`, `score` |
| `violations` | violation cited at an inspection | `inspection_id`, `violation_code`, `critical_flag` |

Reminder: lower score = better. 0-13 = A, 14+ = "failed" the surprise (initial) inspection.

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

# find the P1 database from wherever this notebook is run
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "P1 - NYC Restaurant Grades").is_dir())
DB = REPO / "P1 - NYC Restaurant Grades" / "data" / "processed" / "nyc_inspections.duckdb"
if not DB.exists():
    raise FileNotFoundError("Run `python run_all.py` inside the P1 folder first; it builds the database.")

con = duckdb.connect(str(DB), read_only=True)
sql = lambda q: con.sql(q).df()      # run a query, get a pandas DataFrame back
print("duckdb", duckdb.__version__)
sql("SELECT table_name, estimated_size AS rows FROM duckdb_tables()")

duckdb 1.5.6


,table_name,rows
0,inspections,135864
1,restaurants,25744
2,violations,294914


## The order SQL actually runs a query (why you can't use an alias in `WHERE`)

You write: `SELECT … FROM … JOIN … WHERE … GROUP BY … HAVING … ORDER BY … LIMIT`

The database runs: **`FROM` / `JOIN` → `WHERE` → `GROUP BY` → `HAVING` → window functions → `SELECT` → `DISTINCT` → `ORDER BY` → `LIMIT`**

So:
- `WHERE` runs before `SELECT`: an alias defined in `SELECT` doesn't exist yet there.
- `WHERE` filters **rows**; `HAVING` filters **groups** (after aggregation).
- Window functions run after `GROUP BY`/`HAVING`, so you can't put them in `WHERE`. Compute them in a CTE, then filter in the outer query (Q3).

---
## Q1 — Joins + aggregation + `HAVING`

> **Which 10 cuisines have the worst average score at the initial (surprise) cycle inspection? Only include cuisines with at least 500 such inspections. Also show the % that failed (score 14+).**

**Plan:** filter to initial cycle inspections → join to `restaurants` for the cuisine → group by cuisine → `HAVING` for the minimum count → order.

In [2]:
q1 = """
WITH initial AS (                                   -- 1. only scored surprise inspections
    SELECT inspection_id, camis, score
    FROM inspections
    WHERE program = 'Cycle Inspection'
      AND kind = 'Initial Inspection'
      AND score IS NOT NULL
)
SELECT r.cuisine,
       COUNT(*)                                                          AS n_inspections,
       ROUND(AVG(ini.score), 1)                                          AS avg_score,
       ROUND(100.0 * AVG(CASE WHEN ini.score >= 14 THEN 1 ELSE 0 END), 1) AS pct_failed
FROM initial AS ini
JOIN restaurants AS r ON r.camis = ini.camis       -- 2. many inspections -> one restaurant
GROUP BY r.cuisine                                  -- 3. one row per cuisine
HAVING COUNT(*) >= 500                              -- 4. filter GROUPS, not rows
ORDER BY avg_score DESC                             -- 5. alias is fine here: ORDER BY runs after SELECT
LIMIT 10
"""
sql(q1)

,cuisine,n_inspections,avg_score,pct_failed
0,Indian,917,20.0,54.6
1,Korean,743,18.9,55.5
2,Delicatessen,965,18.8,51.5
3,"Latin (Cuban, Dominican, Puerto Rican, South &...",2766,18.7,52.7
4,Caribbean,2042,18.6,52.2
5,Thai,762,18.6,51.3
6,Spanish,1708,18.4,52.1
7,Jewish/Kosher,906,17.9,48.0
8,Asian,955,17.4,47.7
9,Pizza/Italian,1486,17.4,48.0


**Check the join didn't change the row count** (the roadmap's "know what each join does to your row count"): an inner join of inspections → restaurants is many-to-one, so it must keep every inspection.

In [3]:
sql("""
SELECT (SELECT COUNT(*) FROM inspections
        WHERE program = 'Cycle Inspection' AND kind = 'Initial Inspection' AND score IS NOT NULL) AS before_join,
       (SELECT COUNT(*) FROM inspections i JOIN restaurants r USING (camis)
        WHERE program = 'Cycle Inspection' AND kind = 'Initial Inspection' AND score IS NOT NULL) AS after_join
""")

,before_join,after_join
0,70147,70147


**Common mistakes:** `WHERE COUNT(*) >= 500` (aggregates aren't allowed in `WHERE`); forgetting `score IS NOT NULL` (`AVG` ignores NULLs, but `COUNT(*)` doesn't, so `n` would be inflated); `AVG(score >= 14)` works in DuckDB/Postgres but not everywhere, so `CASE WHEN` is the portable form.

---
## Q2 — Window function with a partition: `LAG`

> **How long does the city wait before a restaurant's next surprise inspection, depending on how the previous one went? Group the previous initial score into A (0-13), B range (14-27) and C range (28+).**

**Plan:** for each restaurant, order its initial inspections by date; `LAG()` over a window **partitioned by restaurant** gives the previous score and date on the same row.

In [4]:
q2 = """
WITH initial AS (
    SELECT camis, inspection_date, score
    FROM inspections
    WHERE program = 'Cycle Inspection' AND kind = 'Initial Inspection' AND score IS NOT NULL
),
with_prev AS (
    SELECT camis, inspection_date, score,
           LAG(score)           OVER (PARTITION BY camis ORDER BY inspection_date) AS prev_score,
           LAG(inspection_date) OVER (PARTITION BY camis ORDER BY inspection_date) AS prev_date
    FROM initial
)
SELECT CASE WHEN prev_score <= 13 THEN '1. previous = A (0-13)'
            WHEN prev_score <= 27 THEN '2. previous = B range (14-27)'
            ELSE                       '3. previous = C range (28+)' END   AS previous_result,
       COUNT(*)                                                           AS n_pairs,
       MEDIAN(date_diff('day', prev_date, inspection_date))               AS median_days_between,
       ROUND(AVG(score), 1)                                               AS avg_score_now
FROM with_prev
WHERE prev_score IS NOT NULL            -- the first inspection of each restaurant has no previous one
GROUP BY previous_result
ORDER BY previous_result
"""
sql(q2)

,previous_result,n_pairs,median_days_between,avg_score_now
0,1. previous = A (0-13),26098,381.0,14.4
1,2. previous = B range (14-27),15439,235.0,18.5
2,3. previous = C range (28+),6073,217.0,21.0


**Reading it:** after an A the next surprise visit comes about a year later (median 381 days); after a B- or C-range score, after about 7-8 months. This is the city's risk-based schedule, and it's why P1 lists **inspection timing** as a limitation.

**Why `PARTITION BY camis` matters:** without it, `LAG` would take the previous row of the *whole table*, i.e. another restaurant's inspection. Quick proof:

In [5]:
sql("""
WITH initial AS (
    SELECT camis, inspection_date, score FROM inspections
    WHERE program = 'Cycle Inspection' AND kind = 'Initial Inspection' AND score IS NOT NULL)
SELECT COUNT(*) FILTER (WHERE prev_camis_no_partition <> camis) AS rows_where_lag_crosses_restaurants
FROM (SELECT camis, LAG(camis) OVER (ORDER BY camis, inspection_date) AS prev_camis_no_partition FROM initial)
""")

,rows_where_lag_crosses_restaurants
0,22536


That is one wrong comparison for **every restaurant's first inspection**: its "previous" row would be the last inspection of a *different* restaurant.

---
## Q3 — Ranking within groups: `RANK() OVER (PARTITION BY …)` + filter in an outer query

> **In each borough, which 3 cuisines have the highest failure rate at initial inspections? Only cuisines with at least 100 initial inspections in that borough.**

**Plan:** aggregate to borough × cuisine → rank *within each borough* → keep ranks 1-3. The rank can't be filtered in the same `SELECT` (window functions run after `WHERE`), hence the extra CTE.

In [6]:
q3 = """
WITH initial AS (
    SELECT r.boro, r.cuisine, i.score
    FROM inspections AS i
    JOIN restaurants AS r USING (camis)
    WHERE i.program = 'Cycle Inspection' AND i.kind = 'Initial Inspection' AND i.score IS NOT NULL
),
by_cuisine AS (
    SELECT boro, cuisine,
           COUNT(*)                                          AS n,
           AVG(CASE WHEN score >= 14 THEN 1.0 ELSE 0.0 END)  AS fail_rate
    FROM initial
    GROUP BY boro, cuisine
    HAVING COUNT(*) >= 100
),
ranked AS (
    SELECT *,
           RANK() OVER (PARTITION BY boro ORDER BY fail_rate DESC) AS rnk   -- restarts at 1 in every borough
    FROM by_cuisine
)
SELECT boro, rnk, cuisine, n, ROUND(100 * fail_rate, 1) AS pct_failed
FROM ranked
WHERE rnk <= 3                                                             -- filter AFTER the window is computed
ORDER BY boro, rnk
"""
sql(q3)

,boro,rnk,cuisine,n,pct_failed
0,BRONX,1,Spanish,359,57.4
1,BRONX,2,"Latin (Cuban, Dominican, Puerto Rican, South &...",569,50.8
2,BRONX,3,Caribbean,338,48.5
3,BROOKLYN,1,Caribbean,1002,55.3
4,BROOKLYN,2,Indian,186,53.8
5,BROOKLYN,3,Spanish,373,53.1
6,MANHATTAN,1,Delicatessen,394,67.8
7,MANHATTAN,2,Thai,368,59.8
8,MANHATTAN,3,"Latin (Cuban, Dominican, Puerto Rican, South &...",582,59.6
9,QUEENS,1,Indian,269,54.6


**`ROW_NUMBER` vs `RANK` vs `DENSE_RANK`** (asked in interviews): for values 60, 55, 55, 50 they give `1,2,3,4` / `1,2,2,4` / `1,2,2,3`. `RANK` keeps ties and can return more than 3 rows per group; `ROW_NUMBER` always returns exactly 3, picking among ties arbitrarily unless you add a tie-breaker to `ORDER BY`.

**Caution on interpretation:** these are failure *rates* with a few hundred inspections each. The differences between ranks 2 and 3 are well inside sampling noise, so don't over-read the order.

---
## Bonus — P1's headline result in SQL (`LEAD` + named `WINDOW`)

The roadmap's Week 8 task 4: *do one analysis in both SQL and pandas and compare.* This reproduces P1's main table: how each A was earned, and the fail rate at the next surprise inspection.

In [7]:
q4 = """
WITH cyc AS (
    SELECT camis, inspection_date, kind, score
    FROM inspections
    WHERE program = 'Cycle Inspection'
      AND kind IN ('Initial Inspection', 'Re-inspection')
      AND score IS NOT NULL
),
cyc_next AS (                                  -- the cycle inspection right after each one
    SELECT *,
           LEAD(kind)            OVER w AS next_kind,
           LEAD(score)           OVER w AS next_row_score,
           LEAD(inspection_date) OVER w AS next_row_date
    FROM cyc
    WINDOW w AS (PARTITION BY camis ORDER BY inspection_date)
),
initials AS (                                  -- each initial inspection + the restaurant's NEXT initial
    SELECT *,
           LEAD(score) OVER (PARTITION BY camis ORDER BY inspection_date) AS next_initial_score
    FROM cyc_next
    WHERE kind = 'Initial Inspection'
),
a_grades AS (
    SELECT camis, next_initial_score,
           CASE WHEN score <= 13 THEN 'A at initial' ELSE 'A after re-inspection' END AS route
    FROM initials
    WHERE score <= 13                                                   -- A straight away
       OR (score >= 14 AND next_kind = 'Re-inspection'                  -- or failed, then...
           AND date_diff('day', inspection_date, next_row_date) <= 120  -- ...re-inspected in the same cycle
           AND next_row_score <= 13)                                    -- ...and scored an A
)
SELECT route,
       COUNT(*)                   AS n_a_grades,
       COUNT(next_initial_score)  AS n_followed,
       ROUND(100.0 * AVG(CASE WHEN next_initial_score >= 14 THEN 1.0 ELSE 0.0 END)
             FILTER (WHERE next_initial_score IS NOT NULL), 2) AS pct_fail_next
FROM a_grades
GROUP BY route
ORDER BY route DESC
"""
sql_result = sql(q4)
sql_result

,route,n_a_grades,n_followed,pct_fail_next
0,A at initial,41297,26098,33.13
1,A after re-inspection,18599,15092,51.63


Why the next initial inspection can come from a window over **initials only**: an A-after-re-inspection is always the row straight after its failed initial, so "the next initial after the re-inspection" is the same as "the next initial after the failed initial".

Compare with the pandas numbers saved by P1:

In [8]:
import json
key = json.loads((REPO / "P1 - NYC Restaurant Grades" / "results" / "key_numbers.json").read_text())
pandas_result = pd.DataFrame({
    "route": ["A at initial", "A after re-inspection"],
    "pct_fail_next_pandas": [round(100 * key["fail_next_A_at_initial"][0], 2),
                             round(100 * key["fail_next_A_after_reinspection"][0], 2)],
})
check = sql_result.merge(pandas_result, on="route", validate="one_to_one")
check["match"] = (check["pct_fail_next"] - check["pct_fail_next_pandas"]).abs() < 0.01
check

,route,n_a_grades,n_followed,pct_fail_next,pct_fail_next_pandas,match
0,A at initial,41297,26098,33.13,33.13,True
1,A after re-inspection,18599,15092,51.63,51.63,True


**Which was clearer?** (the write-up the roadmap asks for)

- **SQL was clearer for the "previous/next row" logic.** `LEAD(...) OVER (PARTITION BY camis ORDER BY date)` says exactly what it means, and a named `WINDOW` keeps three `LEAD`s readable. In pandas the same idea needs `groupby().shift(-1)` plus a backward fill for "next initial", which is harder to read.
- **pandas was clearer for everything after the table**: bootstrap confidence intervals, robustness splits and figures. SQL gives the point estimates, not the uncertainty.
- **Rule of thumb:** shape and filter the data in SQL, close to where it lives; do the statistics and charts in pandas.

In [9]:
con.close()